In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from time import time

# Add TimeSeries root directory to path
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

timeseries_root = os.path.abspath(os.path.join(current_dir, '..', '..', '..'))
print(f"TimeSeries root: {timeseries_root}")

if timeseries_root not in sys.path:
    sys.path.insert(0, timeseries_root)

from TimeSeriesSRC.Model.model import pmodel
from TimeSeriesSRC.Model.pmodsim import func_pmodsim
from TimeSeriesSRC.Model.estimate import estimate  # Correct function name

# Set random seed
np.random.seed(42)

print("\n[OK] Libraries imported successfully")

# Test results storage
test_results = {}

Current directory: D:\Deep Learning Book\All_repo_source_file\Packages\TimeSeries\TimeSeriesSRC\Examples\NoteBooks
TimeSeries root: D:\Deep Learning Book\All_repo_source_file\Packages\TimeSeries

[OK] Libraries imported successfully


# Time Series Models - Comprehensive Test Suite

This notebook tests all model types:
- **ARX**: AutoRegressive with eXogenous inputs
- **ARMAX**: ARX with Moving Average
- **ARMA**: AutoRegressive Moving Average
- **BJTF**: Box-Jenkins Transfer Function

Each test:
1. Creates a model with known parameters
2. Simulates data
3. Estimates parameters from the data
4. Compares estimated vs true parameters
5. Reports PASS/FAIL status

In [2]:
print("="*70)
print("TEST 1: ARX MODEL")
print("="*70)

start_time = time()

try:
    # Parameters (from testarx.m)
    na = 2
    nb = [1, 1, 2]
    delay = [1, 2, 3]
    
    # True parameters
    a_true = np.array([-1.0, 0.25])
    b_true = [
        np.array([1.0, 1.0]),
        np.array([2.0, 3.0]),
        np.array([1.0, -1.0, 0.25])
    ]
    
    # Create true model
    pmoda = pmodel('arx', na=na, nb=nb, delay=delay)
    pmoda.a[0] = a_true
    pmoda.b = b_true
    
    # Generate data
    n = 2000
    u = np.random.randn(3, n)
    e = np.random.randn(n) * 0.5
    y = func_pmodsim(pmoda, e, u)
    
    print(f"[OK] Generated {n} samples")
    
    # Estimate
    pmodb = pmodel('arx', na=na, nb=nb, delay=delay)
    pmodb.estimParams.show = 50
    pmodb.estimParams.epochs = 100
    
    pmod_est, trec, stat = estimate(pmodb, y, u)
    
    # Compare
    params_est = pmod_est.getmX()
    params_true = pmoda.getmX()
    sse = np.sum((params_est - params_true)**2)
    
    print(f"\n[RESULT] Sum of Squared Errors: {sse:.8f}")
    
    # Success criterion
    success = sse < 0.01
    
    test_results['ARX'] = {
        'status': 'PASS' if success else 'FAIL',
        'sse': sse,
        'time': time() - start_time
    }
    
    status = "[PASS]" if success else "[FAIL]"
    print(f"\n{status} ARX Model Test")
    
except Exception as e:
    print(f"\n[ERROR] {str(e)}")
    test_results['ARX'] = {'status': 'ERROR', 'error': str(e)}

print(f"Time elapsed: {time() - start_time:.2f}s")
print("="*70)

TEST 1: ARX MODEL
[OK] Generated 2000 samples
Epoch 0/100 Time 0.016111373901367188 PMODMSE 73.06828802126324/0 Gradient 151019.07091878768/0.0001 mu 0.001/10000000000.0
4.966529147486212e-07 0.0001
Epoch 2/100 Time 0.047040462493896484 PMODMSE 0.26372127258103484/0 Gradient 4.966529147486212e-07/0.0001 mu 1e-05/10000000000.0
ESTIMLM, Minimum gradient reached, performance goal was not met.



[RESULT] Sum of Squared Errors: 0.00105467

[PASS] ARX Model Test
Time elapsed: 0.05s


In [3]:
print("\n" + "="*70)
print("TEST 2: ARMAX MODEL")
print("="*70)

start_time = time()

try:
    # Parameters
    na = 1
    nb = [1]
    nc = 1
    delay = [0]
    
    # True parameters
    a_true = np.array([0.7])
    b_true = [np.array([1.5, 0.8])]
    c_true = np.array([0.5])
    
    # Create and simulate
    pmoda = pmodel('armax', na=na, nb=nb, nc=nc, delay=delay)
    pmoda.a[0] = a_true
    pmoda.b = b_true
    pmoda.c[0] = c_true
    
    n = 1000
    u = np.random.randn(1, n)
    e = np.random.randn(n) * 0.3
    y = func_pmodsim(pmoda, e, u)
    
    print(f"[OK] Generated {n} samples")
    
    # Estimate
    pmodb = pmodel('armax', na=na, nb=nb, nc=nc, delay=delay)
    pmodb.estimParams.show = 50
    pmodb.estimParams.epochs = 100
    
    pmod_est, trec, stat = estimate(pmodb, y, u)
    
    # Compare
    params_est = pmod_est.getmX()
    params_true = pmoda.getmX()
    sse = np.sum((params_est - params_true)**2)
    
    print(f"\n[RESULT] Sum of Squared Errors: {sse:.8f}")
    
    success = sse < 0.05
    
    test_results['ARMAX'] = {
        'status': 'PASS' if success else 'FAIL',
        'sse': sse,
        'time': time() - start_time
    }
    
    status = "[PASS]" if success else "[FAIL]"
    print(f"\n{status} ARMAX Model Test")
    
except Exception as e:
    print(f"\n[ERROR] {str(e)}")
    test_results['ARMAX'] = {'status': 'ERROR', 'error': str(e)}

print(f"Time elapsed: {time() - start_time:.2f}s")
print("="*70)


TEST 2: ARMAX MODEL
[OK] Generated 1000 samples
Input may not be zero mean sequences.
Epoch 0/100 Time 0.0059986114501953125 PMODMSE 2.8459788782801754/0 Gradient 2004.964033823605/0.0001 mu 0.001/10000000000.0
5.834478180720757e-05 0.0001
Epoch 9/100 Time 0.05700230598449707 PMODMSE 0.0829565556684493/0 Gradient 5.834478180720757e-05/0.0001 mu 1.0000000000000006e-12/10000000000.0
ESTIMLM, Minimum gradient reached, performance goal was not met.



[RESULT] Sum of Squared Errors: 0.00868860

[PASS] ARMAX Model Test
Time elapsed: 0.06s


In [4]:
print("\n" + "="*70)
print("TEST 3: ARMA MODEL")
print("="*70)

start_time = time()

try:
    # Parameters (simplified from testarma.m)
    nc = [1]
    nd = [1]
    diff = [0]
    per = []
    
    # True parameters
    c_true = [np.array([0.65])]
    d_true = [np.array([0.75])]
    
    # Create and simulate
    pmoda = pmodel('arma', nc=nc, nd=nd, diff=diff, per=per)
    pmoda.c = c_true
    pmoda.d = d_true
    
    n = 1000
    e = np.random.randn(n) * 0.5
    y = func_pmodsim(pmoda, e)
    
    print(f"[OK] Generated {n} samples")
    
    # Estimate
    pmodb = pmodel('arma', nc=nc, nd=nd, diff=diff, per=per)
    pmodb.estimParams.show = 50
    pmodb.estimParams.epochs = 100
    
    pmod_est, trec, stat = estimate(pmodb, y)  # Using 'estimate' function
    
    # Compare
    params_est = pmod_est.getmX()
    params_true = pmoda.getmX()
    sse = np.sum((params_est - params_true)**2)
    
    print(f"\n[RESULT] Sum of Squared Errors: {sse:.8f}")
    
    success = sse < 0.05
    
    test_results['ARMA'] = {
        'status': 'PASS' if success else 'FAIL',
        'sse': sse,
        'time': time() - start_time
    }
    
    status = "[PASS]" if success else "[FAIL]"
    print(f"\n{status} ARMA Model Test")
    
except Exception as e:
    print(f"\n[ERROR] {str(e)}")
    test_results['ARMA'] = {'status': 'ERROR', 'error': str(e)}

print(f"Time elapsed: {time() - start_time:.2f}s")
print("="*70)


TEST 3: ARMA MODEL
[OK] Generated 1000 samples
Epoch 0/100 Time 0.004995107650756836 PMODMSE 0.23689283959578697/0 Gradient 26.798464413896024/0.0001 mu 0.001/10000000000.0
5.044574743204075e-05 0.0001
Epoch 5/100 Time 0.03187131881713867 PMODMSE 0.23325089202285001/0 Gradient 5.044574743204075e-05/0.0001 mu 1.0000000000000004e-05/10000000000.0
ESTIMLM, Minimum gradient reached, performance goal was not met.



[RESULT] Sum of Squared Errors: 0.00122024

[PASS] ARMA Model Test
Time elapsed: 0.03s


In [5]:
print("\n" + "="*70)
print("TEST 4: BJTF MODEL")
print("="*70)

start_time = time()

try:
    # Parameters (simplified from testbjtf.m)
    nb = [1]
    nc = [1]
    nd = [1]
    nf = [1]
    delay = [1]
    diff = [0]
    per = []
    
    # True parameters
    b_true = [np.array([1.0, 0.8])]
    c_true = [np.array([0.5])]
    d_true = [np.array([0.7])]
    f_true = [np.array([0.3])]
    
    # Create and simulate
    pmoda = pmodel('bjtf', nb=nb, nc=nc, nd=nd, nf=nf, 
                   delay=delay, diff=diff, per=per)
    pmoda.b = b_true
    pmoda.c = c_true
    pmoda.d = d_true
    pmoda.f = f_true
    
    n = 500
    u = np.random.randn(1, n)
    e = np.random.randn(n) * 0.3
    y = func_pmodsim(pmoda, e, u)
    
    print(f"[OK] Generated {n} samples")
    
    # Estimate
    pmodb = pmodel('bjtf', nb=nb, nc=nc, nd=nd, nf=nf,
                   delay=delay, diff=diff, per=per)
    pmodb.estimParams.show = 50
    pmodb.estimParams.epochs = 100
    
    pmod_est, trec, stat = estimate(pmodb, y, u)
    
    # Compare
    params_est = pmod_est.getmX()
    params_true = pmoda.getmX()
    sse = np.sum((params_est - params_true)**2)
    
    print(f"\n[RESULT] Sum of Squared Errors: {sse:.8f}")
    
    success = sse < 0.1
    
    test_results['BJTF'] = {
        'status': 'PASS' if success else 'FAIL',
        'sse': sse,
        'time': time() - start_time
    }
    
    status = "[PASS]" if success else "[FAIL]"
    print(f"\n{status} BJTF Model Test")
    
except Exception as e:
    print(f"\n[ERROR] {str(e)}")
    test_results['BJTF'] = {'status': 'ERROR', 'error': str(e)}

print(f"Time elapsed: {time() - start_time:.2f}s")
print("="*70)


TEST 4: BJTF MODEL
[OK] Generated 500 samples
Input may not be zero mean sequences.
Epoch 0/100 Time 0.003947734832763672 PMODMSE 1.2115351425901595/0 Gradient 666.2704043765036/0.0001 mu 0.001/10000000000.0
7.118475497969535e-05 0.0001
Epoch 12/100 Time 0.04194784164428711 PMODMSE 0.10245049619396177/0 Gradient 7.118475497969535e-05/0.0001 mu 1.0000000000000006e-11/10000000000.0
ESTIMLM, Minimum gradient reached, performance goal was not met.



[RESULT] Sum of Squared Errors: 0.01060470

[PASS] BJTF Model Test
Time elapsed: 0.04s


D:\Deep Learning Book\All_repo_source_file\Packages\TimeSeries\TimeSeriesSRC\Model\pmodmse.py:74: RuntimeWarning: overflow encountered in multiply
  res =e *m* e


## Visualization of Results

## Test Summary

In [6]:
print("\n" + "="*70)
print("TEST SUMMARY")
print("="*70)

total_tests = len(test_results)
passed = sum(1 for r in test_results.values() if r['status'] == 'PASS')
failed = sum(1 for r in test_results.values() if r['status'] == 'FAIL')
errors = sum(1 for r in test_results.values() if r['status'] == 'ERROR')

print(f"\nTotal Tests: {total_tests}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")
print(f"Errors: {errors}")

print("\nDetailed Results:")
print("-" * 70)
print(f"{'Model':<15} {'Status':<10} {'SSE':<15} {'Time (s)':<10}")
print("-" * 70)

for model, result in test_results.items():
    status = result['status']
    sse = f"{result.get('sse', 0):.6f}" if 'sse' in result else 'N/A'
    elapsed = f"{result.get('time', 0):.2f}" if 'time' in result else 'N/A'
    print(f"{model:<15} {status:<10} {sse:<15} {elapsed:<10}")

print("="*70)

# Overall result
if errors > 0:
    print("\n[ERROR] Some tests encountered errors!")
elif failed > 0:
    print("\n[FAIL] Some tests failed!")
else:
    print("\n[SUCCESS] All tests passed!")

print("\nNote: These tests validate that:")
print("  1. Models can be created successfully")
print("  2. Data simulation works correctly")
print("  3. Parameter estimation converges")
print("  4. Estimated parameters are close to true values")


TEST SUMMARY

Total Tests: 4
Passed: 4
Failed: 0
Errors: 0

Detailed Results:
----------------------------------------------------------------------
Model           Status     SSE             Time (s)  
----------------------------------------------------------------------
ARX             PASS       0.001055        0.05      
ARMAX           PASS       0.008689        0.06      
ARMA            PASS       0.001220        0.03      
BJTF            PASS       0.010605        0.04      

[SUCCESS] All tests passed!

Note: These tests validate that:
  1. Models can be created successfully
  2. Data simulation works correctly
  3. Parameter estimation converges
  4. Estimated parameters are close to true values
